In [1]:
import pandas as pd
from data_gen import generate_timeseries_multi
from feat import compute_features

In [2]:
df = generate_timeseries_multi()
feat = compute_features(df)
print(feat.tail())

                   SPX         VIX        DXY      US10Y  SPX_ret_1d  \
2022-09-22  112.892754  100.290740  89.901901  73.244617   -0.002811   
2022-09-23  114.922212  100.515201  89.125286  72.162557    0.017977   
2022-09-24  115.658683  100.262259  89.016922  71.919228    0.006408   
2022-09-25  114.998065   98.795407  88.761851  71.302522   -0.005712   
2022-09-26  115.656524   98.603442  87.364637  70.328090    0.005726   

            VIX_ret_1d  DXY_ret_1d  US10Y_ret_1d  SPX_ret_5d  VIX_ret_5d  ...  \
2022-09-22    0.000817    0.018925     -0.006625   -0.041924    0.011083  ...   
2022-09-23    0.002238   -0.008638     -0.014773   -0.004373    0.024865  ...   
2022-09-24   -0.002516   -0.001216     -0.003372    0.004490    0.016541  ...   
2022-09-25   -0.014630   -0.002865     -0.008575    0.005611   -0.003263  ...   
2022-09-26   -0.001943   -0.015741     -0.013666    0.021602   -0.016021  ...   

            DXY_ret_21d  US10Y_ret_21d   SPX_z63   VIX_z63   DXY_z63  \
2022-09-

In [3]:
feat.index

DatetimeIndex(['2020-01-01', '2020-01-02', '2020-01-03', '2020-01-04',
               '2020-01-05', '2020-01-06', '2020-01-07', '2020-01-08',
               '2020-01-09', '2020-01-10',
               ...
               '2022-09-17', '2022-09-18', '2022-09-19', '2022-09-20',
               '2022-09-21', '2022-09-22', '2022-09-23', '2022-09-24',
               '2022-09-25', '2022-09-26'],
              dtype='datetime64[us]', length=1000, freq='D')

In [4]:
out = pd.DataFrame(index=feat.index)

# Risk regime
risk_on = (feat["SPX_ret_21d"] > 0) & (feat["VIX_z63"] < 0.5)
risk_off = (feat["SPX_ret_21d"] < 0) & (feat["VIX_z63"] > 1)

out["risk_regime"] = "Mixed"
out.loc[risk_on, "risk_regime"] = "Risk-On"
out.loc[risk_off, "risk_regime"] = "Risk-Off"

print(out["risk_regime"].tail(20))

2022-09-07       Mixed
2022-09-08    Risk-Off
2022-09-09    Risk-Off
2022-09-10    Risk-Off
2022-09-11    Risk-Off
2022-09-12    Risk-Off
2022-09-13    Risk-Off
2022-09-14    Risk-Off
2022-09-15    Risk-Off
2022-09-16    Risk-Off
2022-09-17    Risk-Off
2022-09-18       Mixed
2022-09-19    Risk-Off
2022-09-20    Risk-Off
2022-09-21    Risk-Off
2022-09-22    Risk-Off
2022-09-23    Risk-Off
2022-09-24    Risk-Off
2022-09-25       Mixed
2022-09-26       Mixed
Freq: D, Name: risk_regime, dtype: str


In [5]:
 # Liquidity regime
liq_tight = (feat["DXY_ret_21d"] > 0) & (feat["US10Y_ret_21d"] > 0)
liq_ease = (feat["DXY_ret_21d"] < 0)

out["liquidity_regime"] = "Neutral"
out.loc[liq_tight, "liquidity_regime"] = "Tightening"
out.loc[liq_ease, "liquidity_regime"] = "Easing"

print(out[["risk_regime", "liquidity_regime"]].tail())

           risk_regime liquidity_regime
2022-09-22    Risk-Off           Easing
2022-09-23    Risk-Off           Easing
2022-09-24    Risk-Off           Easing
2022-09-25       Mixed           Easing
2022-09-26       Mixed           Easing


In [6]:
final = pd.concat([feat, out], axis=1)

print(final.tail())

                   SPX         VIX        DXY      US10Y  SPX_ret_1d  \
2022-09-22  112.892754  100.290740  89.901901  73.244617   -0.002811   
2022-09-23  114.922212  100.515201  89.125286  72.162557    0.017977   
2022-09-24  115.658683  100.262259  89.016922  71.919228    0.006408   
2022-09-25  114.998065   98.795407  88.761851  71.302522   -0.005712   
2022-09-26  115.656524   98.603442  87.364637  70.328090    0.005726   

            VIX_ret_1d  DXY_ret_1d  US10Y_ret_1d  SPX_ret_5d  VIX_ret_5d  ...  \
2022-09-22    0.000817    0.018925     -0.006625   -0.041924    0.011083  ...   
2022-09-23    0.002238   -0.008638     -0.014773   -0.004373    0.024865  ...   
2022-09-24   -0.002516   -0.001216     -0.003372    0.004490    0.016541  ...   
2022-09-25   -0.014630   -0.002865     -0.008575    0.005611   -0.003263  ...   
2022-09-26   -0.001943   -0.015741     -0.013666    0.021602   -0.016021  ...   

             SPX_z63   VIX_z63   DXY_z63  US10Y_z63  SPX_z252  VIX_z252  \
2022-

In [10]:
final.to_parquet("/workspaces/codespaces-blank/data/features_test.parquet")

In [11]:
tickers = {
    # macro
    "SPX": "SPX",
    "DXY": "DXY",
    "US10Y": "US10Y",
    "VIX": "VIX",

    # crypto
    #"BTC": "BTC-USD",
    #"ETH": "ETH-USD",
}

tickers.values()

dict_values(['SPX', 'DXY', 'US10Y', 'VIX'])

In [22]:
last = final.iloc[-1]

def fmt(x):
    return f"{x:.2f}"

print("\n=== MACRO DASHBOARD ===\n")

print("Asset  |    Close |     1d%     5d%    21d% | Z-score")
print()

for asset in tickers.values():
    print(
        f"{asset:6} | "
        f"{fmt(last[asset]):>8} | "
        f"{fmt(last[f'{asset}_ret_1d']*100):>6}% "
        f"{fmt(last[f'{asset}_ret_5d']*100):>6}% "
        f"{fmt(last[f'{asset}_ret_21d']*100):>6}% | "
        f"z={fmt(last[f'{asset}_z63'])}"
    )

print("\nRisk:", last["risk_regime"])
print("Liquidity:", last["liquidity_regime"])


=== MACRO DASHBOARD ===

Asset  |    Close |     1d%     5d%    21d% | Z-score

SPX    |   115.66 |   0.57%   2.16%   0.29% | z=-0.76
DXY    |    87.36 |  -1.57%  -0.98%  -2.36% | z=-2.25
US10Y  |    70.33 |  -1.37%  -4.62%  -5.95% | z=-1.40
VIX    |    98.60 |  -0.19%  -1.60%   2.13% | z=0.87

Risk: Mixed
Liquidity: Easing


In [35]:
last = final.iloc[-1]

def fmt_num(x, width=8, decimals=2):
    if pd.isna(x):
        return " " * width
    return f"{x:>{width}.{decimals}f}"

def fmt_pct(x, width=7, decimals=2):
    if pd.isna(x):
        return " " * width
    return f"{x * 100:>{width}.{decimals}f}"

print("=" * 50)
print("DASHBOARD".ljust(31) + f"{final.index[-1]}")
print("=" * 50)

print(
    f"RISK: {last['risk_regime']:<10}"
    f" LIQ: {last['liquidity_regime']:<14}"
)
print("-" * 50)

sections = {
    "MACRO": ["SPX", "US10Y", "VIX"],
    "FX": ["DXY"],
    #"CRYPTO": ["BTC", "ETH"],
}

for section, assets in sections.items():
    print(f"\n[{section}]")
    print("ASSET         LAST     1D%     5D%    21D%     Z63")
    print("-" * 50)

    for asset in assets:
        print(
            f"{asset:<8} "
            f"{fmt_num(last[asset], 9)} "
            f"{fmt_pct(last[f'{asset}_ret_1d'], 7)} "
            f"{fmt_pct(last[f'{asset}_ret_5d'], 7)} "
            f"{fmt_pct(last[f'{asset}_ret_21d'], 7)} "
            f"{fmt_num(last[f'{asset}_z63'], 7)}"
        )

DASHBOARD                      2022-09-26 00:00:00
RISK: Mixed      LIQ: Easing        
--------------------------------------------------

[MACRO]
ASSET         LAST     1D%     5D%    21D%     Z63
--------------------------------------------------
SPX         115.66    0.57    2.16    0.29   -0.76
US10Y        70.33   -1.37   -4.62   -5.95   -1.40
VIX          98.60   -0.19   -1.60    2.13    0.87

[FX]
ASSET         LAST     1D%     5D%    21D%     Z63
--------------------------------------------------
DXY          87.36   -1.57   -0.98   -2.36   -2.25
